# Proportion Reg 28 and Reg 30 Reports

In [3]:
# libraries, libraries!

import time

start_time_prpn = time.time()
start_time = time.time()
print(f"Importing libraries ...")

from datetime import datetime
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from constants import pthPy, pthReports
from utilities import timediff, prior_month_end

print(f"Importing libraries completed: {timediff(start_time, time.time())}\n")

Importing libraries ...
Importing libraries completed: 0.0sec



In [4]:
# get report variables

start_time = time.time()
print("Getting report variables ...")

# report variables
sr = pd.read_excel(pthPy, sheet_name="arc", index_col=None, usecols="AL", nrows=4)

prn = sr.iloc[2, 0].upper()  # fund name
rpt_type = sr.iloc[3, 0].capitalize()  # Reg28 or Reg30 report
print(prn, rpt_type)

k = sr.iloc[1, 0]  # overriden report date
rptDate = k.date() if k == k else prior_month_end().date()
fnm = os.path.join(
    pthReports, f"{prn.upper()} {rpt_type} {rptDate.strftime('%d%b%Y')}.xlsx"
)
print(f" Fund: {prn}\n Report type: {rpt_type}\n Date: {rptDate}\n Path: {fnm}")

# get the underlying fund and recalibrate holdings percentages
src = pd.read_excel(fnm)
src["End Market Value"] = src["End Market Value"] / src["End Market Value"].sum()
src["Closing Exposure PA"] = (
    src["Closing Exposure PA"] / src["Closing Exposure PA"].sum()
)

fnds = pd.read_excel(
    pthPy, sheet_name="arc", index_col=None, header=0, usecols="AI:AK"
).dropna(subset=["Name"])

s = "" if len(fnds["Name"]) == 1 else "s"
print(
    f"\n {len(fnds['Name'])} investor fund{s}:\n  {(', ').join(list(fnds['Name']))}\n"
)

print(f"Getting report variables completed: {timediff(start_time, time.time())}", "\n")

Getting report variables ...
PSIF Reg30
 Fund: PSIF
 Report type: Reg30
 Date: 2025-12-31
 Path: \\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Reg28 and Reg30 Reporting\PSIF Reg30 31Dec2025.xlsx

 1 investor fund:
  Glencore Medical Scheme

Getting report variables completed: 3.6sec 



In [5]:
# create the client proportional reports

start_time = time.time()
print("Creating client proportional reports ...")

mdd0 = []
eed0 = []
mdd1 = []
eed1 = []
for i in tqdm(range(len(fnds))):
    # create the proportional client holdings
    client = src.copy()
    client["Entity Name"] = f"{fnds.at[i, 'Name']} ({prn})"
    client["End Market Value"] = client["End Market Value"] * fnds.iloc[i, 2]
    client["End Market Value"] = client["End Market Value"].round(decimals=2)
    client["Closing Exposure PA"] = client["Closing Exposure PA"] * fnds.iloc[i, 2]
    client["Closing Exposure PA"] = client["Closing Exposure PA"].round(decimals=2)

    # get rounding differences before eliminating them
    mdd0.append(client["End Market Value"].sum())
    eed0.append(client["Closing Exposure PA"].sum())

    # eliminate rounding differences
    client.at[0, "End Market Value"] = (
        client.at[0, "End Market Value"]
        - client["End Market Value"].sum()
        + fnds.iloc[i, 2]
    )
    client.at[0, "Closing Exposure PA"] = (
        client.at[0, "Closing Exposure PA"]
        - client["Closing Exposure PA"].sum()
        + fnds.iloc[i, 2]
    )

    # get rounding differences after eliminating them
    mdd1.append(client["End Market Value"].sum())
    eed1.append(client["Closing Exposure PA"].sum())

    # save the client fund in the reporting folder
    kl = fnds.iloc[i, 0]
    initials = "".join(word[0].upper() for word in kl.split()) + "(" + prn + ")"
    mn = f"{kl} ({prn}) {rpt_type} {rptDate.strftime('%d%b%Y')}.xlsx"
    client.to_excel(
        os.path.join(pthReports, mn),
        index=False,
        sheet_name=f"{initials} {rpt_type} {rptDate.strftime('%d%b%Y')}",
    )

print(f"{timediff(start_time, time.time())} creating client proportional reports")

# open the reporting folder where the files are saved
os.startfile(os.path.realpath(pthReports))

Creating client proportional reports ...


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.91s/it]

2.9sec creating client proportional reports


In [6]:
print(os.path.join(pthReports, mn))
print(f"\n{timediff(start_time_prpn, time.time())} roundtrip time for {len(fnds['Name'])} investor fund{s}")

\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Reg28 and Reg30 Reporting\Glencore Medical Scheme (PSIF) Reg30 31Dec2025.xlsx

6.8sec roundtrip time for 1 investor fund


In [7]:
# # for review, present client fund NAV rounding numbers

# start_time = time.time()
# print("Creating NAV difference columns ...")

# fnds["MV_diff_before"] = (fnds["Value"] - mdd0).round(2)
# fnds["EE_diff_before"] = (fnds["Value"] - eed0).round(2)
# fnds["MV_diff_after"]  = (fnds["Value"] - mdd1).round(2)
# fnds["EE_diff_after"]  = (fnds["Value"] - eed1).round(2)

# fnds["Value"] = fnds["Value"].apply(lambda x: "{:,.2f}".format(x))

# print(
#     f"Creating NAV difference columns completed: {timediff(start_time, time.time())}",
#     "\n",
# )
# print(f"Roundtrip time: {timediff(start_time0, time.time())}", "\n")

# fnds[
#     [
#         "Name",
#         "Reference",
#         "Value",
#         "MV_diff_before",
#         "EE_diff_before",
#         "MV_diff_after",
#         "EE_diff_after",
#     ]
# ]

In [8]:
# !jupyter nbconvert --to script proportioner.ipynb # convert from .ipynb to .py